In [1]:
import pandas as pd

In [4]:
cs = pd.read_csv('../data/true_sources_final.csv')

In [6]:
cs = pd.read_csv(
    "../data/true_sources_final.csv",
    dtype={"source_id": "string"},
)

cs["source_id"] = cs["source_id"].str.strip()

# Keep a record of duplicate input rows for inspection
duplicate_rows = cs[
    cs.duplicated("source_id", keep=False)
].sort_values("source_id")

duplicate_rows.to_csv("../data/duplicate_source_rows.csv", index=False)

# Temporary ranking columns
cs["_syd_success_num"] = pd.to_numeric(
    cs["syd_success"],
    errors="coerce",
)

cs["_numax_err_num"] = pd.to_numeric(
    cs["numax_sig"],
    errors="coerce",
)

cs["_preferred_syd"] = cs["_syd_success_num"].eq(2)

# Sort so the preferred row is first for each source_id
cs = (
    cs.sort_values(
        by=["source_id", "_preferred_syd", "_numax_err_num"],
        ascending=[True, False, True],
        na_position="last",
        kind="stable",
    )
    .drop_duplicates("source_id", keep="first")
    .drop(columns=[
        "_syd_success_num",
        "_numax_err_num",
        "_preferred_syd",
    ])
    .reset_index(drop=True)
)

print(f"Duplicate rows removed: {len(duplicate_rows) - len(cs[cs['source_id'].isin(duplicate_rows['source_id'])])}")
print(f"Remaining stars: {len(cs)}")

Duplicate rows removed: 10
Remaining stars: 2581


In [ ]:
from pathlib import Path
import pandas as pd


# Adjust these paths
base_dir = Path("/Users/z5214708/BASTA/BASTA/final_runs")
master_file = base_dir / "data" / "true_sources_final.csv"
output_dir = base_dir / "output"


def normalise_id(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def read_basta_results(filepath):
    """Read a BASTA results.ascii file."""
    with filepath.open() as file:
        header = file.readline().strip()

    column_names = header.lstrip("#").split()

    results = pd.read_csv(
        filepath,
        sep=r"\s+",
        names=column_names,
        skiprows=1,
        dtype={"starid": "string"},
    )

    results["starid"] = normalise_id(results["starid"])

    return results


def write_basta_results(results, filepath):
    """Write a clean BASTA-style ASCII file."""
    with filepath.open("w") as file:
        file.write("# " + " ".join(results.columns) + "\n")
        results.to_csv(
            file,
            sep=" ",
            header=False,
            index=False,
            na_rep="-999.999",
        )


# ---------------------------------------------------------------------
# Select the preferred row for each source_id
# ---------------------------------------------------------------------

master = pd.read_csv(
    master_file,
    dtype={"source_id": "string"},
)

master["source_id"] = normalise_id(master["source_id"])

master["_syd_success"] = pd.to_numeric(
    master["syd_success"],
    errors="coerce",
)

master["_numax_err"] = pd.to_numeric(
    master["numax_err"],
    errors="coerce",
)

selected_input = (
    master
    .sort_values(
        by=["source_id", "_syd_success", "_numax_err"],
        ascending=[True, False, True],
        na_position="last",
        kind="stable",
    )
    .drop_duplicates("source_id", keep="first")
)

selected_ids = set(
    selected_input["source_id"]
    .dropna()
)


# ---------------------------------------------------------------------
# Process all 16 BASTA runs
# ---------------------------------------------------------------------

conflict_summary = []

for table in range(1, 5):
    for subset in "abcd":

        run_dir = output_dir / f"table{table}_{subset}"

        for filename in [
            "results.ascii",
            "results_distance.ascii",
        ]:

            input_file = run_dir / filename

            if not input_file.exists():
                continue

            results = read_basta_results(input_file)

            # Keep only source IDs whose preferred input row was selected
            filtered = results[
                results["starid"].isin(selected_ids)
            ].copy()

            # Identify duplicate output rows
            duplicate_rows = filtered[
                filtered.duplicated("starid", keep=False)
            ].copy()

            conflict_ids = []

            for starid, group in duplicate_rows.groupby(
                "starid",
                sort=False,
            ):
                values_only = group.drop(columns="starid")

                # True if duplicate output rows are not identical
                if len(values_only.drop_duplicates()) > 1:
                    conflict_ids.append(starid)

            # Save all duplicate output rows for inspection
            if not duplicate_rows.empty:
                review_file = run_dir / (
                    input_file.stem + "_duplicates_for_review.csv"
                )
                duplicate_rows.to_csv(review_file, index=False)

            # Produce one row per starid
            clean_results = filtered.drop_duplicates(
                "starid",
                keep="first",
            )

            clean_file = run_dir / (
                input_file.stem + "_clean.ascii"
            )

            write_basta_results(clean_results, clean_file)

            conflict_summary.append({
                "run": run_dir.name,
                "file": filename,
                "original_rows": len(results),
                "clean_rows": len(clean_results),
                "duplicate_rows": len(duplicate_rows),
                "conflicting_starids": len(conflict_ids),
            })

            print(
                f"{run_dir.name}/{filename}: "
                f"{len(results)} -> {len(clean_results)} rows; "
                f"{len(conflict_ids)} conflicting IDs"
            )


summary = pd.DataFrame(conflict_summary)
summary.to_csv(
    output_dir / "postprocessing_summary.csv",
    index=False,
)